# Assignment 5: RAG Agent
**Objective**: Implement a RAG Agent that can answer multi-step questions using a different data source than the previous assignments.

## 1. Setup and Installs

In [ ]:
%pip install -qU langchain langchain-core "langchain[openai]" langchain-text-splitters langchain-community bs4 langchain-huggingface sentence-transformers

In [ ]:
import os
import bs4
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_openai import ChatOpenAI

## 2. Model and Vector Store Initialization

In [ ]:
try:
    from google.colab import userdata
    openrouter_api_key = userdata.get("OPENROUTER_API_KEY")
except ImportError:
    openrouter_api_key = os.environ.get("OPENROUTER_API_KEY", "YOUR_OPENROUTER_API_KEY_HERE")

os.environ["OPENROUTER_API_KEY"] = openrouter_api_key

model_nemotron3_nano = ChatOpenAI(
    model="nvidia/nemotron-3-nano-30b-a3b:free",
    temperature=0,
    base_url="https://openrouter.ai/api/v1",
    api_key=openrouter_api_key,
)

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-mpnet-base-v2"
)

vector_store = InMemoryVectorStore(embeddings)

## 3. Indexing a New Data Source
We use **Lilian Weng's 'Prompt Engineering'** blog post as our new data source.

In [ ]:
bs4_strainer = bs4.SoupStrainer(class_=("post-title", "post-header", "post-content"))
loader = WebBaseLoader(
    web_paths=("https://lilianweng.github.io/posts/2023-03-15-prompt-engineering/",),
    bs_kwargs={"parse_only": bs4_strainer},
)
docs = loader.load()

print(f"Loaded {len(docs)} document(s).")
if docs:
    print(f"Total characters: {len(docs[0].page_content)}")

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    add_start_index=True,
)
all_splits = text_splitter.split_documents(docs)

print(f"Split blog post into {len(all_splits)} sub-documents.")

document_ids = vector_store.add_documents(documents=all_splits)
print(f"Stored {len(document_ids)} chunks in the vector store.")

## 4. Retrieval Tool Definition
We create a tool for the agent to retrieve information from the indexed documents.

In [ ]:
from typing import Literal
from langchain.tools import tool

@tool(response_format="content_and_artifact")
def retrieve_context(
    query: str,
    section: Literal["beginning", "middle", "end"] = "middle"
):
    """Retrieve information to help answer a query."""
    retrieved_docs = vector_store.similarity_search(query, k=2)
    serialized = "\n\n".join(
        (f"Source: {doc.metadata}\n"
        f"Content: {doc.page_content}")
        for doc in retrieved_docs
    )
    content, artifact = serialized, retrieved_docs
    return content, artifact

## 5. RAG Agent Creation

In [ ]:
try:
    # Standard method based on the lesson's API
    from langchain.agents import create_agent
    agent = create_agent(
        model=model_nemotron3_nano,
        tools=[retrieve_context],
        system_prompt=(
            "You have access to a tool that retrieves context from a blog post about Prompt Engineering. "
            "Use the tool to help answer user queries. Do not guess; rely on the retrieved info."
        )
    )
except ImportError:
    # Fallback to langgraph's prebuilt agent if 'create_agent' doesn't exist directly in langchain.agents
    from langgraph.prebuilt import create_react_agent
    agent = create_react_agent(
        model=model_nemotron3_nano,
        tools=[retrieve_context],
        state_modifier=(
            "You have access to a tool that retrieves context from a blog post about Prompt Engineering. "
            "Use the tool to help answer user queries. Do not guess; rely on the retrieved info."
        )
    )

## 6. Testing the Agent with a Multi-Step Question
We ask a question that requires multiple steps of reasoning and context retrieval.

In [ ]:
from langchain.messages import HumanMessage

query = (
    "What is Zero-Shot prompting?\n\n"
    "Once you get the answer, look up what Few-Shot prompting is and summarize how they differ."
)

result = agent.invoke({"messages": [HumanMessage(query)]})

for msg in result["messages"]:
    msg.pretty_print()